# 00 · Quickstart: loading the data

A full run takes ~5 min (two sampled scans of `interactions`, the catalog and the monthly
counts; the last two are cached for the other notebooks).

Start Jupyter from `notebooks/` with your AWS profile:
`AWS_PROFILE=admin uv run jupyter lab`. The helpers live in `src/steam_eda/` (see README).

| Source | Loader | What |
|---|---|---|
| Iceberg marts (Glue `steam_marts`) | `load_mart`, `iter_mart` | `interactions`, `game_features`, `game_tags`, `game_details`, `user_features`, `lkp_*` |
| Raw scraped parquet (`raw-steam-data-<acct>`) | `load_raw`, `scan_raw` | `reviews`, `games`, `game_tags`: every field Steam returns, also the ones the marts drop |

In [ ]:
import os

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from steam_eda import (
    Sample,
    binned_rate,
    cached,
    categorical_rate,
    decode,
    feature_report,
    game_catalog,
    gini,
    iter_mart,
    load_mart,
    load_raw,
    lookup,
    lorenz,
    mart_schema,
    mart_table,
    raw_files,
    review_counts,
    tag_profile_similarity,
    with_user_history,
)
from steam_eda.plots import loglog_hist, rate_plot, setup

setup()

## Marts: schema and size

Snapshot summaries give row counts without reading data.

In [ ]:
for name in ["interactions", "game_features", "game_tags", "game_details", "user_features"]:
    snapshot = mart_table(name).current_snapshot()
    print(
        f"{name:15s} {int(snapshot.summary['total-records']):>13,d} rows  "
        f"{int(snapshot.summary['total-files-size']) / 1e9:6.2f} GB"
    )
mart_schema("interactions")

## Loading patterns

- `where`: Iceberg row filter, pushed down. Skips whole files, so it's the cheap way to cut a
  time window. **Not representative**: only that window.
- `sample=Sample.by_key("user_id", f)`: whole histories of a fraction `f` of the users,
  representative of users. Reads every file (only the selected columns), keeps ~f of the rows.
- `sample=Sample.rows(f)`: uniform rows, representative of rows. Breaks user histories.
- `filter=<polars expr>`: any row filter, applied per batch.
- `limit=n`: the first n kept rows in file order. Fast, **biased** (file order).

Always pass `columns`: the list columns dominate the bytes.

In [ ]:
COLUMNS = ["user_id", "game_idx", "timestamp", "is_positive"]

recent = load_mart("interactions", columns=COLUMNS, where="timestamp >= '2026-09-01T00:00:00'")
peek = load_mart("interactions", columns=COLUMNS, limit=10_000)
users = load_mart("interactions", columns=COLUMNS, sample=Sample.by_key("user_id", 0.001))
rows = load_mart(
    "interactions", columns=COLUMNS, sample=Sample.rows(0.001), filter=pl.col("is_positive")
)
for name, df in {"recent": recent, "peek": peek, "users": users, "rows": rows}.items():
    print(
        f"{name:7s} {df.height:>9,d} rows  {df['user_id'].n_unique():>8,d} users  "
        f"{df['timestamp'].min()} → {df['timestamp'].max()}"
    )

## Games: ids → names, and the one-row-per-game catalog

List columns hold dense ids (`0` = padding, `1` = OOV); `decode` adds the names.
`game_catalog()` streams the 115M-row `game_features` mart and the interactions (~2–3 min,
~9 GB peak), so it is wrapped in `cached` (`notebooks/.cache/`, `refresh=True` after a pipeline
run).

In [ ]:
catalog = cached("catalog", game_catalog)
decode(catalog.head(5), ["game_genres", "game_tags"]).select(
    "game_name",
    "game_genres_names",
    "game_tags_names",
    "game_price",
    "release_year",
    "game_coming_soon",
    "reviews",
)

## Raw parquet

Append-only scrapes: a review or game scraped twice has two rows (dedupe on `rec_id` /
`appid`). `load_raw` skips the review texts unless you ask for `review`. `last_files=n` reads
the n newest files: quick, but only the latest run's games.

In [ ]:
print(f"{len(raw_files('reviews')):,d} raw review files")
raw = load_raw(
    "reviews",
    columns=[
        "rec_id",
        "author_id",
        "appid",
        "voted_up",
        "playtime_at_review",
        "language",
        "timestamp_created",
    ],
    last_files=20,
)
raw.unique("rec_id").describe()

## Big aggregations: stream with `iter_mart`

When the result is small but the input isn't (counts over all 115M interactions), aggregate
batch by batch and combine.

In [ ]:
def monthly() -> pl.DataFrame:
    parts = [
        b.group_by(month=pl.col("timestamp").dt.truncate("1mo")).agg(
            reviews=pl.len(), positives=pl.col("is_positive").sum()
        )
        for b in iter_mart("interactions", columns=["timestamp", "is_positive"])
    ]
    return (
        pl.concat(parts).group_by("month").agg(pl.col("reviews", "positives").sum()).sort("month")
    )


months = cached("monthly_interactions", monthly)
months.tail(6)